# Transformer Teardown: Build a Minimal Llama Toolkit

In [the last Transformer Teardown post](https://stickshift.github.io/articles/llama1/), we tore down and rebuilt Meta's Llama 3 LLM, gaining a hands-on, close-up view of the machinery inside a state-of-the-art generative transformer. The goal of this post is to leverage what we learned to create a minimal toolkit for experimenting with Llama models. While there are plenty of open source Llama implementations out there, they're often complicated and overloaded with configuration settings to the point that the main ideas are completely obscured. Building your own collection of minimal Llama components will do wonders for your knowledge of Transformer fundamentals. Not only that, but you'll walk away with a valuable toolkit for running your own research experiments. We'll be sure to revisit this idea in a future post.

# Setup

## Imports

In [1]:
import json
import logging
from pathlib import Path
from random import sample
from sys import stdout
from typing import Any, Iterator, Mapping, NamedTuple, Sequence

from llama_models.llama3.api.tokenizer import Tokenizer
from llama_models.llama3.reference_impl.model import RMSNorm
import numpy as np
from llama.tools import default_arg, device as torch_device
import torch
from torch import Tensor, nn
from torch.nn.functional import silu, softmax

## Jupyter HTML Hacks

In [2]:
%%html
<style>
a {
    color: rgb(29, 66, 115) !important;
    font-weight: bold;
}
</style>

# Text Generation Pipeline

{ref}`transformer-pipeline-fig` illustrates the main stages of a generative Transformer pipeline. Raw text is split into a sequence of tokens. Tokens are mapped to embeddings. Embeddings are transformed through multiple context layers. Semantic embeddings are used as features to predict the next token in the sequence. Finally, the predicted token is fed back into the pipeline and the process repeats.

```{figure} resources/transformer-pipeline.svg
:label: transformer-pipeline-fig
:width: 80%

Transformer Pipeline
```

# Generator Design Pattern

In the last post, we walked through each step of the text generation pipeline in {ref}`transformer-pipeline-fig`. Our main goal in this post is to assemble the steps into reusable building blocks following common design patterns established by popular frameworks such as Hugging Face's [transformers](https://github.com/huggingface/transformers) library.

{ref}`generator-design-pattern-fig` illustrates the main components of the Generator design pattern. The `Embeddings` and `Layer` components are bundled together in a general purpose `Model` component that is reused across modeling tasks. `Generator` and `Head` are task-specific components that apply the generic capabilities of Model to different problems. This modular design allows you to easily apply the same underlying Transformer model to problems like causal language modeling, sequence classification, and question answering tasks by swapping out the `Generator` and `Head` components.

```{figure} resources/generator-design-pattern.svg
:label: generator-design-pattern-fig
:width: 80%

Generator Design Pattern
```

While many Transformers share these general concepts, the implementations are architecture-specific. For example, if you look at the GPT2, Gemma2, and Llama implementations in Hugging Face’s [transformers](https://github.com/huggingface/transformers) library, you’ll see each one implements their own version of each component.

Following this pattern, we'll implement our own collection of Llama components based on the minimal implementation from [Transformer Teardown: Llama 3.1](https://stickshift.github.io/articles/llama1/).

# Model Config

[The Llama website](https://www.llama.com/) provides multiple model *checkpoints*, representing different sizes and model configurations. Each checkpoint includes a json configuration file (`params.json`) and pre-trained model weights (`consolidated.00.pth`).

Here, we define a `ModelConfig` data structure and `load_config` function to load the checkpoint-specific hyperparameters from `params.json`. Examples include the number of parameters in each layer (`d_model`), the number of layers (`n_layers`), and the number of attention heads (`n_heads`).

In [3]:
class ModelConfig(NamedTuple):
    """Llama3 model config."""

    checkpoint_path: Path

    vocab_size: int

    d_model: int

    d_head: int

    d_ffn: int

    n_layers: int

    n_heads: int

    n_kv_heads: int

    rms_norm_eps: float

    rope_theta: float

In [4]:
def load_config(checkpoint_name: str, **kwargs) -> ModelConfig:
    """Load Llama3 config from checkpoint params.json."""

    # Build checkpoint_path
    checkpoints_path = Path("~/.llama/checkpoints").expanduser()
    checkpoint_path = checkpoints_path / checkpoint_name

    # Load hyperparameters
    hparams_path = checkpoint_path / "params.json"
    hparams = json.loads(hparams_path.read_text())

    # Calculate d_ffn from 8/3 * d_model rounded to nearest multiple_of
    d_model = hparams["dim"]
    ffn_dim_multiplier = hparams["ffn_dim_multiplier"]
    multiple_of = hparams["multiple_of"]
    d_ffn = int(8 / 3 * d_model * ffn_dim_multiplier)
    d_ffn = multiple_of * ((d_ffn + multiple_of - 1) // multiple_of)

    data = {
        "checkpoint_path": checkpoint_path,
        "vocab_size": hparams["vocab_size"],
        "d_model": hparams["dim"],
        "n_layers": hparams["n_layers"],
        "rms_norm_eps": hparams["norm_eps"],
        "n_heads": hparams["n_heads"],
        "d_head": int(hparams["dim"] / hparams["n_heads"]),
        "n_kv_heads": hparams["n_kv_heads"],
        "rope_theta": hparams["rope_theta"],
        "d_ffn": d_ffn,
    }

    # Override with kwargs
    data |= kwargs

    return ModelConfig(**data)

Let's load the config for the Llama3.2-3B checkpoint as an example.

In [5]:
model_config = load_config("Llama3.2-3B")
model_config._asdict()

{'checkpoint_path': PosixPath('/Users/andrewyoung/.llama/checkpoints/Llama3.2-3B'),
 'vocab_size': 128256,
 'd_model': 3072,
 'd_head': 128,
 'd_ffn': 8192,
 'n_layers': 28,
 'n_heads': 24,
 'n_kv_heads': 8,
 'rms_norm_eps': 1e-05,
 'rope_theta': 500000.0}

# Checkpoint

Next, we load the pre-trained weights from `consolidated.00.pth` into a `Checkpoint` data structure that maps component names to weights. Each of the components we define later will load their weights from this structure.

In [ ]:
# Model checkpoint in-memory data structure
Checkpoint = Mapping[str, Any]

def load_checkpoint(config: ModelConfig) -> Checkpoint:
    """Load model checkpoint from disk."""
    return torch.load(
        config.checkpoint_path / "consolidated.00.pth",
        weights_only=True,
    )

In [ ]:
checkpoint = load_checkpoint(model_config)

In [ ]:
# Display sample of keys in checkpoint
sample(sorted({k for k in checkpoint}), k=5)

# Embeddings

Next, we'll implement a `LlamaEmbeddings` component that maps token ids to token embeddings.

In [ ]:
class LlamaEmbeddings(nn.Embedding):
    """Map token_ids to token embeddings."""

    def __init__(self, config: ModelConfig, checkpoint: Checkpoint):
        # Configure embeddings
        super().__init__(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model,
            device=config.device,
        )

        # Load parameters
        self.load_state_dict({"weight": checkpoint["tok_embeddings.weight"]})

# Context Layers

Next, we'll implement a `LlamaLayer` component that implements the attention and feedforward network blocks in a single decoder layer. 

This where all the Transformer magic happens, and there is a lot going on here. For details on the implementation, please refer to [Transformer Teardown: Llama 3.1](https://stickshift.github.io/articles/llama1/) where we walk through each step one by one. For the purposes of this post, the important takeaway is simply that the logic is arranged into reusable building blocks.

## Rotary Position Embeddings (RoPE)

In [ ]:
def rope_frequencies(config: ModelConfig, n: int):
    """Compute RoPE cos and sin rotation matrices."""
    # Hyperparameters
    base = config.rope_theta
    d = config.d_head

    # Calculate thetas
    i = torch.arange(d // 2, device=config.device)
    thetas = base ** (-2 * i / d)

    # Duplicate each theta, e.g. [theta_0, theta_1] -> [theta_0, theta_0, theta_1, theta_1]
    thetas = thetas.repeat_interleave(2)

    # Repeat thetas for each position from 0 to n and stack in an (n, d_head) matrix
    theta_stack = torch.stack([m * thetas for m in range(n)])

    # Apply cos, sin
    r_cos = torch.cos(theta_stack)
    r_sin = torch.sin(theta_stack)

    # Sanity check
    assert r_cos.shape[0] == n and r_cos.shape[1] == config.d_head
    assert r_sin.shape[0] == n and r_sin.shape[1] == config.d_head

    return r_cos, r_sin


def rope_swap(x):
    """Maps [x0, x1, x2, x3] -> [-x1, x0, -x3, x2]."""
    # Preserve original shape
    s = x.shape

    # Split into pairs, swap, and restore shape
    x = x.reshape(-1, 2).flip(-1).view(s)

    # Multiply every even index along the last dimension by -1
    #   e.g. [x0, x1, x2, x3] -> [-x0, x1, -x2, x3]
    x[..., ::2] *= -1

    return x


def rope_rotate(x, r_cos, r_sin):
    """Rotate embeddings using RoPE transform."""
    return (x * r_cos) + (rope_swap(x) * r_sin)

## LlamaLayer

In [ ]:
class LlamaLayer(nn.Module):
    """Llama decoder layer."""

    def __init__(self, config: ModelConfig, checkpoint: Checkpoint, layer_id: int):
        super().__init__()

        self.config = config
        self.layer_id = layer_id

        # Attention normalization
        self.normalize_attention = RMSNorm(config.d_model, config.rms_norm_eps)
        self.normalize_attention.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.attention_norm.weight"],
        })

        # Query projection
        self.w_q = nn.Linear(
            in_features=config.d_model,
            out_features=config.n_heads * config.d_head,
            bias=False,
        )
        self.w_q.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.attention.wq.weight"],
        })

        # Key projection
        self.w_k = nn.Linear(
            in_features=config.d_model,
            out_features=config.n_kv_heads * config.d_head,
            bias=False,
        )
        self.w_k.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.attention.wk.weight"],
        })

        # Value projection
        self.w_v = nn.Linear(
            in_features=config.d_model,
            out_features=config.n_kv_heads * config.d_head,
            bias=False,
        )
        self.w_v.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.attention.wv.weight"],
        })

        # Attention output projection
        self.w_a = nn.Linear(
            in_features=config.d_model,
            out_features=config.d_model,
            bias=False,
        )
        self.w_a.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.attention.wo.weight"],
        })

        # FFN normalization
        self.normalize_ffn = RMSNorm(config.d_model, config.rms_norm_eps)
        self.normalize_ffn.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.ffn_norm.weight"],
        })

        # SwiGLU FFN
        self.w_h = nn.Linear(
            in_features=config.d_model,
            out_features=config.d_ffn,
            bias=False,
        )
        self.w_h.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.feed_forward.w3.weight"],
        })

        self.w_g = nn.Linear(
            in_features=config.d_model,
            out_features=config.d_ffn,
            bias=False,
        )
        self.w_g.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.feed_forward.w1.weight"],
        })

        # FFN output projection
        self.w_f = nn.Linear(
            in_features=config.d_ffn,
            out_features=config.d_model,
            bias=False,
        )
        self.w_f.load_state_dict({
            "weight": checkpoint[f"layers.{layer_id}.feed_forward.w2.weight"],
        })

    def forward(self, x: Tensor, r_cos: Tensor, r_sin: Tensor) -> Tensor:        
        # ---------------------------------------------------------------------
        # Attention
        # ---------------------------------------------------------------------
        
        residual = x

        # Normalize attention inputs
        x = self.normalize_attention(x)

        # Project embeddings to query, key, value spaces
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        # Split attention heads
        q = self._split_heads(q, self.config.n_heads)
        k = self._split_heads(k, self.config.n_kv_heads)
        v = self._split_heads(v, self.config.n_kv_heads)

        # Expand key/value groups
        reps = self.config.n_heads // self.config.n_kv_heads
        k = k.repeat_interleave(reps, dim=0)
        v = v.repeat_interleave(reps, dim=0)

        # Encode positions by rotating queries and keys
        q = rope_rotate(q, r_cos, r_sin)
        k = rope_rotate(k, r_cos, r_sin)

        # Compute masked attention bias M
        n = len(x)
        mask = torch.ones(n, n, dtype=torch.bool, device=self.config.device).tril(diagonal=0)
        m = torch.zeros(n, n, device=self.config.device).masked_fill_(mask.logical_not(), float("-inf"))

        # Compute attention for all heads in parallel
        a = softmax(q @ k.transpose(-2, -1) / np.sqrt(self.config.d_head) + m, dim=-1) @ v

        # Combine attention heads
        a = self._combine_heads(a)

        # Project attention representations back to model space
        a = self.w_a(a)

        # Combine attention representations with residual embeddings
        x = residual + a

        # ---------------------------------------------------------------------
        # FFN
        # ---------------------------------------------------------------------

        residual = x

        # Normalize FFN inputs
        x = self.normalize_ffn(x)

        # Apply SwiGLU transform
        f = silu(self.w_g(x)) * self.w_h(x)

        # Project FFN representations back to model space
        f = self.w_f(f)

        # Combine FFN representations with residual embeddings
        x = residual + f

        return x

    def _split_heads(self, x: Tensor, n_heads: int):
        """Split attention heads."""
        return x.view(-1, n_heads, self.config.d_head).transpose(-3, -2)

    def _combine_heads(self, x):
        """Combine attention heads."""
        return x.transpose(-3, -2).contiguous().view(-1, int(self.config.n_heads * self.config.d_head))

# Model

Next, we'll implement a `LlamaModel` component that bundles the embeddings and context layers together. This `model = embeddings + context layers` design is a common pattern in Hugging Face and other Transformer implementations.

In [ ]:
class LlamaModel(nn.Module):
    """Bundles embeddings and context layers together."""

    def __init__(self, config: Config, checkpoint: Checkpoint):
        super().__init__()

        # Config
        self.config = config
        
        # Embeddings
        self.embeddings = LlamaEmbeddings(config, checkpoint)

        # Transformer layers
        self.layers = nn.ModuleList(
            LlamaLayer(config, checkpoint, layer_id) for layer_id in range(config.n_layers)
        )

    def forward(self, token_ids: Tensor) -> Tensor:
        # Compute cos and sin rotation matrices once for entire sequence
        r_cos, r_sin = rope_frequencies(self.config, len(token_ids))
        
        # Map tokens to embeddings
        x = self.embeddings(token_ids)

        # Transform token embeddings to semantic embeddings
        for layer in self.layers:
            x = layer(x, r_cos, r_sin)

        return x

# Head

Next, we'll implement `LlamaHead` and `LlamaCausalLMHead` modules. `LlamaHead` is an abstract module that serves as a base class for task-specific head layers. `LlamaHead` provides common functions such as loading checkpoint weights and projecting the semantic embeddings back to token space. `LlamaCausalLMHead` extends `LlamaHead` to implement "causal language modeling" (aka. next token prediction) based on temperature, top_k, and top_p token sampling.

In [ ]:
class LlamaHead(nn.Module):
    """Abstract Llama head layer."""

    def __init__(self, config: Config, checkpoint: Checkpoint):
        super().__init__()

        self.config = config

        # Head normalization
        self.normalize_head = RMSNorm(config.d_model, config.rms_norm_eps).to(config.device)
        self.normalize_head.load_state_dict({
            "weight": checkpoint["norm.weight"],
        })

        # Output projection
        self.w_head = nn.Linear(
            in_features=config.d_model,
            out_features=config.vocab_size,
            bias=False,
            device=config.device,
        )
        self.w_head.load_state_dict({
            "weight": checkpoint["output.weight"],
        })

    def _project_outputs(self, x: Tensor) -> Tensor:
        """Project embeddings to token space."""
        # Normalize head inputs
        x = self.normalize_head(x)

        # Use last embedding to represent the entire sequence
        x = x[-1]

        # Project outputs to token space
        x = self.w_head(x)

        return x

In [ ]:
class LlamaCausalLMHead(LlamaHead):
    """Predicts next token id using temperature, top_k, and top_p token sampling."""

    def forward(self, x: Tensor) -> int:
        # Project semantic embeddings to token space
        x = self._project_outputs(x)

        # If temperature is 0, return the top token
        if self.config.temperature == 0:
            return torch.argmax(x, dim=-1).item()

        # ---------------------------------------------------------------------
        # Temperature
        # ---------------------------------------------------------------------

        # Apply temperature
        x = x / self.config.temperature

        # ---------------------------------------------------------------------
        # Ranking
        # ---------------------------------------------------------------------

        # Convert logits to probabilities
        probs = softmax(x, dim=-1)

        # Sort probabilities in descending order
        probs, indices = probs.sort(descending=True)

        # ---------------------------------------------------------------------
        # Top K
        # ---------------------------------------------------------------------

        # Retain top k tokens
        probs = probs[: self.config.top_k]

        # ---------------------------------------------------------------------
        # Top P
        # ---------------------------------------------------------------------

        # Find cutoff where cumulative probability exceeds top_p
        cumulative_mask = probs.cumsum(dim=-1) > self.config.top_p
        threshold_index = torch.argmax(cumulative_mask).item()

        # Only apply threshold if top_p was exceeded
        if cumulative_mask.any():
            probs = probs[: threshold_index + 1]

        # ---------------------------------------------------------------------
        # Random Selection
        # ---------------------------------------------------------------------

        # Sample from remaining tokens weighted by probability
        sampled_index = torch.multinomial(probs, 1)

        # Convert sampled_index to original logits
        token_id = indices[sampled_index]

        return token_id.item()

# Generator

Next, we'll implement the `LlamaGenerator` module that combines `LlamaModel` with `LlamaCausalLMHead`. Given a sequence of token ids, `LlamaGenerator` starts by predicting the next token in the sequence before feeding the predicted token back into the model in an autoregressive decoding loop. `LlamaGenerator` continues generating new tokens until it predicts a stop token or exceeds the `max_tokens` parameter.

In [ ]:
class LlamaGenerator:
    """Generates text using a Llama autoregressive decoder."""

    def __init__(self, config: Config, stop_tokens: Sequence[int]):
        
        self.config = config
        self.stop_tokens = stop_tokens
        
        # Load checkpoint
        checkpoint = load_checkpoint(config)

        # Model
        self.model = LlamaModel(config, checkpoint).to(config.device)

        # Prediction head
        self.head = LlamaCausalLMHead(config, checkpoint).to(config.device)

    def __call__(self, token_ids: Sequence[int]) -> Iterator[int]:
        # Prepare model
        self.model.eval()

        # Make mutable copy of token ids
        token_ids = list(token_ids)
        
        with torch.no_grad():
            # Generate output until we get a stop token or we exceed max_tokens.
            for _ in range(self.config.max_tokens):
                # Load token ids into a tensor
                x = torch.tensor(token_ids, device=self.config.device)

                # Transform token_ids into semantic embeddings
                x = self.model(x)

                # Predict next token
                token_id = self.head(x)

                # Check stopping criteria
                if token_id in self.stop_tokens:
                    break

                # Yield token
                yield token_id

                # Append to end of sequence
                token_ids.append(token_id)

# Pipeline

Finally, we'll combine all of the pieces in a simple end-to-end text generation pipeline.

In [ ]:
def generate_text(tokenizer, generator, prompt: str) -> Iterator[str]:
    # Split prompt into tokens
    token_ids = tokenizer.encode(prompt, bos=True, eos=False, allowed_special="all")

    # Generate tokens
    for token_id in generator(token_ids):

        # Decode token_id
        token = tokenizer.decode([token_id])

        yield token

# Generating Text with Llama 3.2-3B 

In [6]:
model = "Llama3.2-3B"
config = load_config(model)
tokenizer = Tokenizer(str(config.checkpoint_path / "tokenizer.model"))

In [8]:
prompt = "alpha beta gamma"
token_ids = tokenizer.encode(prompt, bos=True, eos=False, allowed_special="all")
token_ids

[128000, 7288, 13746, 22350]

In [ ]:
model = "Llama3.2-3B"
config = load_config(model)

In [ ]:
tokenizer = Tokenizer(str(config.checkpoint_path / "tokenizer.model"))
generator = LlamaGenerator(config, tokenizer.stop_tokens)

In [ ]:
prompt = "Humpty dumpty sat"

stdout.write(prompt)
for token in generate_text(tokenizer, generator, prompt):
    stdout.write(token)

In [ ]:
model = "Llama3.2-3B"
config = load_config(model)

In [ ]:
tokenizer = Tokenizer(str(config.checkpoint_path / "tokenizer.model"))
generator = LlamaGenerator(config, tokenizer.stop_tokens)

In [ ]:
prompt = "Humpty dumpty sat"
token_ids = tokenizer.encode(prompt, bos=True, eos=False, allowed_special="all")

In [ ]:
it = generator(token_ids)

In [ ]:
token_id = next(it)

In [ ]:
token_id

In [ ]:
for token_id in generator(token_ids):
    token = tokenizer.decode([token_id])
    print(token)

In [ ]:
model = "Llama3.2-3B"
config = load_config(model)
checkpoint = load_checkpoint(config)

In [ ]:
# Model
model = LlamaModel(config, checkpoint).to(config.device)

# Prediction head
head = LlamaCausalLMHead(config, checkpoint).to(config.device)

In [ ]:
tokenizer = Tokenizer(str(config.checkpoint_path / "tokenizer.model"))

In [ ]:
prompt = "Humpty dumpty sat"

In [ ]:
token_ids = tokenizer.encode(prompt, bos=True, eos=False, allowed_special="all")
x = torch.tensor(token_ids, device=config.device)

In [ ]:
# Transform token_ids into semantic embeddings
x = model(x)
x

In [ ]:
# Predict next token
token_id = head(x)
token_id

In [ ]:
token = tokenizer.decode([token_id])
token

In [ ]:
tokenizer = Tokenizer(str(config.checkpoint_path / "tokenizer.model"))
embeddings = LlamaEmbeddings(config, checkpoint)
layers = nn.ModuleList(LlamaLayer(config, checkpoint, layer_id) for layer_id in range(config.n_layers))
head = LlamaCausalLMHead(config, checkpoint)

In [ ]:
prompt = "Humpty dumpty sat"

In [ ]:
token_ids = tokenizer.encode(prompt, bos=True, eos=False, allowed_special="all")

In [ ]:
r_cos, r_sin = rope_frequencies(config, len(token_ids))

In [ ]:
# Load token ids into a tensor
x = torch.tensor(token_ids, device=config.device)
x

In [ ]:
x = embeddings(x)
x

In [ ]:
for layer in layers:
    x = layer(x, r_cos, r_sin)
x

In [ ]:
token_id = head(x)
token_id

In [ ]:
token = tokenizer.decode([token_id])
token

In [ ]:
generator = LlamaGenerator(config, stop_tokens=tokenizer.stop_tokens)

In [ ]:
tokenizer = Tokenizer(str(config.checkpoint_path / "tokenizer.model"))
generator = LlamaGenerator(config, stop_tokens=tokenizer.stop_tokens)

In [ ]:
prompt = "Humpy dumpty"
for token in generate_text(tokenizer, generator, prompt):
    stdout.write(token)
stdout.flush()